# 第 7 课：端到端预测管线

目标：把窗口、特征、记忆、检索、候选、反思、基线和反馈串成闭环。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 7.1 先看数据流

```text
历史窗口 → 特征 → Top-K 记忆 → 多候选
         → 物理反思 → 选择结果 → 与恒速基线比较
         → 真值成熟后更新记忆置信度
```

核心源码：[pipeline.py](../src/memcast_uav/pipeline.py)  
文字讲解：[07_end_to_end.md](../tutorial/07_end_to_end.md)

## 7.2 运行 10 个离线测试样本

In [ ]:
from memcast_uav.pipeline import run_demo

result = run_demo(test_limit=10)
result.to_dict()

## 7.3 分字段解释结果

In [ ]:
print("样本数:", result.samples)
print("MemCast式预测 MSE:", result.mse)
print("恒速基线 MSE:", result.baseline_mse)
print("记忆条目:", result.memory_entries)
print("接受候选:", result.accepted_candidates)
print("拒绝候选:", result.rejected_candidates)
print("相对基线改进:", result.baseline_mse - result.mse)

不要只看最终 MSE：如果复杂方法没有优于恒速基线，就不能只凭流程复杂度声称有效。
本教学结果只验证代码链路，不等于复现论文指标。

## 7.4 分块练习：改变测试样本数

In [ ]:
for test_limit in (1, 4, 10):
    trial = run_demo(test_limit=test_limit)
    print(
        f"samples={trial.samples:2d}, "
        f"mse={trial.mse:.6f}, baseline={trial.baseline_mse:.6f}"
    )
# TODO：解释为什么样本太少时，平均误差不稳定。

## 7.5 本课验收

In [ ]:
import math
import subprocess

assert math.isfinite(result.mse)
assert math.isfinite(result.baseline_mse)
assert (
    result.accepted_candidates + result.rejected_candidates
    == 4 * result.samples
)

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_pipeline.py", "-q"],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 6 课](06_confidence.ipynb) · [教程目录](README.md) · [下一课：意图条件预测 →](08_intent.ipynb)